# StreamLoop churn model tuning

**Decision:** optimize recall for churners because missing a customer who is about to leave costs more than offering retention to someone who would have stayed. Precision, F1, accuracy, and ROC AUC remain visible so this business choice is transparent.

## Experiment design and leakage controls

1. Load the official IBM-modeled customer dataset from its required URL.
2. Split train/test before fitting preprocessing or models.
3. Keep numeric coercion, imputation, scaling, and one-hot encoding inside a single scikit-learn `Pipeline`.
4. Evaluate the default baseline on test once.
5. Search only training data: broad `RandomizedSearchCV`, then focused `GridSearchCV`, both with five-fold stratified CV, `scoring="recall"`, and `n_jobs=-1`.
6. Use `GridSearchCV.best_estimator_`, already refitted on all training rows, and evaluate it on test once—without manual refitting.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "src"))
from train import run_experiment, write_report

results = run_experiment()
print(f"rows={results['dataset_rows']:,} train={results['training_rows']:,} test={results['test_rows']:,}")
print("primary metric:", results["primary_metric"])

rows=7,043 train=5,634 test=1,409
primary metric: recall


## Baseline and final test metrics

The default model is a genuine pre-search baseline. The final metrics come from the grid search's automatically refitted estimator. No parameter choice used these test results.

In [2]:
comparison = pd.DataFrame(
    [results["baseline_metrics"], results["final_metrics"]],
    index=["Default baseline", "Tuned final"],
).round(3)
display(comparison)
print("Recall improvement:", round(results["final_metrics"]["recall"] - results["baseline_metrics"]["recall"], 3))

,recall,precision,f1,accuracy,roc_auc
Default baseline,0.559,0.657,0.604,0.806,0.842
Tuned final,0.856,0.443,0.583,0.676,0.816


Recall improvement: 0.297


The recall improvement is large and matches StreamLoop's stated cost asymmetry. Precision and accuracy decline because the class-balanced model flags more potential churners. That trade-off is accepted here, but deployment should pair the score with offer-cost monitoring and threshold review.

## Broad search, focused refinement, and fold stability

The random stage explores 24 combinations over five orders of magnitude for C, two compatible penalties, and optional class weighting. The focused grid tests values around the random winner. Selection considers both mean recall and standard deviation across folds—not only rank.

In [3]:
print("RandomizedSearchCV best mean recall:", round(results["random_best_recall"], 3))
print("GridSearchCV best mean recall:", round(results["grid_best_recall"], 3))
print("Selected fold standard deviation:", round(results["grid_best_std"], 3))
display(pd.DataFrame(results["top_grid_candidates"]))

RandomizedSearchCV best mean recall: 0.837
GridSearchCV best mean recall: 0.837
Selected fold standard deviation: 0.024


,rank_test_score,mean_test_score,std_test_score,params
0,1,0.837458,0.024449,"{'classifier__C': 0.003919406774847221, 'class..."
1,2,0.799331,0.028379,"{'classifier__C': 0.0019597033874236107, 'clas..."
2,3,0.795318,0.016302,"{'classifier__C': 0.0019597033874236107, 'clas..."
3,3,0.795318,0.026789,"{'classifier__C': 0.003919406774847221, 'class..."
4,5,0.793980,0.039832,"{'classifier__C': 0.007838813549694443, 'class..."
5,6,0.791304,0.030914,"{'classifier__C': 0.007838813549694443, 'class..."
6,7,0.509030,0.033605,"{'classifier__C': 0.007838813549694443, 'class..."
7,8,0.481605,0.032009,"{'classifier__C': 0.003919406774847221, 'class..."


The selected candidate has the highest mean CV recall (0.837) and a fold standard deviation of about 0.024. A nearby candidate is slightly more stable (about 0.016) but loses roughly four recall points; that loss is too large for StreamLoop's priority, so the best-mean candidate is retained. Its variability is still comparable with neighboring configurations, not an unstable outlier.

In [4]:
display(pd.Series(results["final_params"], name="selected value").to_frame())
write_report(results, ROOT / "tuning_report.md")
print("tuning_report.md updated from these executed results")

,selected value
classifier__C,0.003919
classifier__class_weight,balanced
classifier__penalty,l1
classifier__solver,liblinear


tuning_report.md updated from these executed results


## Final recommendation

Use the tuned model as the candidate for an operational pilot, not an automatic retention decision. It catches substantially more churners, while the lower precision means retention outreach volume and cost must be monitored. Revisit the classification threshold and offer economics with production feedback before broad rollout.